In terminal do:

python -m venv venv (Create a Python virtual environment)

In PowerShell: venv\Scripts\Activate.ps1

python -m pip install -r requirements.txt

ollama pull llama3:8b

Source: https://www.cohorte.co/blog/using-ollama-with-python-step-by-step-guide

In [1]:
import ollama
import json
import random

In [2]:
from datasets import load_dataset

# Load the dataset to train and test https://huggingface.co/models?other=LLM
dataset = load_dataset("UniqueData/email-spam-classification", split="train")


#plit the data: we use half to train and half to predict
split_data = dataset.train_test_split(test_size=0.1, seed=1)

train_data = split_data["train"]
test_data = split_data["test"]

c:\Boti\!FH\1_szemeszter\AI\Spam detector\test2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def build_examples(train_data, n_spam=5, n_ham=5, seed=1):
    spam = [row for row in train_data if row["type"] == "spam"]
    ham = [row for row in train_data if row["type"] == "not spam"]

    random.seed(seed)
    spam_samples = random.sample(spam, n_spam)
    ham_samples = random.sample(ham, n_ham)

    examples = []
    for row in spam_samples + ham_samples:
        score = 1 if row["type"] == "spam" else 0
        examples.append(
            f"""Email:
\"\"\"{row['text']}\"\"\"
Spam score: {score}
"""
        )
    return "\n".join(examples)

In [4]:
Examples = build_examples(train_data)

SYSTEM_PROMPT = f"""
You are a spam detection system.

Spam score definition:
- Number between 0.0 and 1.0
- 0.0 = definitely not spam
- 1.0 = definitely spam
- Any number between 0.0 and 1.0 represents your confidence
- Only output 0 or 1 if you are fully certain
- Respond ONLY with valid JSON. The reasoning should only be one sentence, under 10 words.
{{
  "spam_score": float,
  "reasoning": string
}}

Below are labeled training examples:

Email:
"Win a free iPhone now!"
Spam score: 0.05

Email:
"Meeting agenda for tomorrow"
Spam score: 0.92

{Examples}

Now analyze the next email.

"""

In [5]:
def check_json(raw_output):
    start = raw_output.find("{")
    end = raw_output.find("}")

    if start != -1 and end != -1 and end > start:
        return raw_output[start:end + 1]

    if end == -1:
        fixed = raw_output.strip() + "\n}"

    if start == -1:
        sc = fixed.find("spam_score")
        fixed = '{\n"' + fixed[sc:]
    
    return fixed

In [6]:
MODEL_NAME = "llama3:8b"

def classify_email(email_text, email_title):
    content = f"Title: {email_title}\n\nBody:\n{email_text}"
    
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Email:\n\"\"\"{content}\"\"\""}
        ]
    )
    
    raw = response["message"]["content"]
    raw = check_json(raw)
    return json.loads(raw)

In [15]:
import pandas as pd
from tqdm import tqdm

results = []

for row in tqdm(test_data):
    pred = classify_email(row["text"], email_title=row["title"])

    score = pred["spam_score"]
    predicted_label = "spam" if score > 0.5 else "not spam"

    results.append({
        "true_label": row["type"],
        "predicted_label": predicted_label,
        "spam_score": score,
        "reasoning": pred["reasoning"],
        "title": row["title"],
        "text": row["text"]
    })

df = pd.DataFrame(results)
df["correct"] = df["true_label"] == df["predicted_label"]

accuracy = df["correct"].mean()
print(f"Test accuracy: {accuracy:.2%}")

100%|██████████| 9/9 [04:34<00:00, 30.54s/it]

Test accuracy: 66.67%


In [17]:
df

,true_label,predicted_label,spam_score,reasoning,title,text,correct
0,not spam,not spam,0.00,This email appears to be a legitimate notifica...,"amazon.com.tr, action needed: Sign-in","\nLogo Image\nSenol Yildirim,\n\nSomeone signe...",True
1,not spam,not spam,0.00,Reward email with no suspicious or malicious c...,You've Earned a Reward from Bard Explorers India,You've received a gift!\nSign in to your Bard ...,True
2,not spam,spam,1.00,"Phishing-like behavior, generic congratulatory...",Congratulations! You have been selected for a ...,Dear Joseph Alex Eze\n \nWe are pleased to inf...,False
3,not spam,not spam,0.00,This is a legitimate verification email from T...,Verify your Twitch email address,"Hey fanat_papicha_nomer_1, please verify your ...",True
4,spam,spam,1.00,The email is offering a job opportunity with u...,"Job Opportunities that pay ?600,000+ and up in...",To cushion the effect of subsidy removal and g...,True
5,not spam,not spam,0.00,This email is a legitimate reminder from the Q...,Come back and keep earning with Quick Pay Survey,Quick Pay Survey® \n \n \n \n \nHello Anggr...,True
6,spam,spam,0.92,The email contains a promotional offer with a ...,"John, here’s a discount for your next safe rid...","Bolt rides are not only affordable, but they a...",True
7,spam,not spam,0.00,The email is a legitimate notification from Di...,Combating scams in the Discogs Marketplace,Discogs has recently detected an increase in s...,False
8,spam,not spam,0.00,The email appears to be a legitimate marketing...,Style with Denim,"Comfortable & Durable\nLove, Bonito\nNEW IN B...",False
